# M5 Forecasting — Accuracy

Predict 28 days of unit sales for Walmart items across stores and states.

**Data files:**
- `sales_train_validation.csv` — daily sales (d_1 … d_1913), item/store metadata
- `sales_train_evaluation.csv` — extended sales (d_1 … d_1941)
- `calendar.csv` — date features, events, SNAP flags
- `sell_prices.csv` — weekly item prices per store
- `sample_submission.csv` — submission template (F1–F28)

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import jax.numpy as jnp
from jax import random

import numpyro
numpyro.set_host_device_count(4)
from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS
import matplotlib.pyplot as plt
from wunkui.models.ssp import kalman_filter_1d

DATA_DIR = Path("../resource/m5-forecasting-accuracy")
HORIZON = 28  # 28-day forecast horizon

## 1. Load Data

In [ ]:
calendar = pd.read_csv(DATA_DIR / "calendar.csv", parse_dates=["date"])
sell_prices = pd.read_csv(DATA_DIR / "sell_prices.csv")
sales_train = pd.read_csv(DATA_DIR / "sales_train_validation.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"calendar:     {calendar.shape}")
print(f"sell_prices:  {sell_prices.shape}")
print(f"sales_train:  {sales_train.shape}")
print(f"submission:   {submission.shape}")

In [ ]:
# Quick look at each table
display(calendar.head(3))
display(sell_prices.head(3))
display(sales_train.iloc[:3, :10])  # first 10 columns to keep it readable
display(submission.head(3))

## 2. Build sales matrix (n_series, n_steps)

In [ ]:
day_cols = sorted(
    [c for c in sales_train.columns if c.startswith("d_")],
    key=lambda x: int(x.split("_")[1]),
)

sales_matrix = sales_train[day_cols].values
series_ids = sales_train["id"].values

print(f"sales_matrix: {sales_matrix.shape}  (n_series={sales_matrix.shape[0]}, n_steps={sales_matrix.shape[1]})") 
print(f"series_ids:   {series_ids[:3]} …")
print(f"day_cols:     {day_cols[:3]} … {day_cols[-3:]}")

## 3. Fit Model — State Space (one series at a time)

Uses our `kalman_filter_1d` with a **local level + weekly Fourier seasonality** design matrix and NUTS sampling.

In [ ]:
def _make_fourier_series(
    n_steps: int, period: float, order: int,
) -> jnp.ndarray:
    """Build Fourier design matrix of shape (n_steps, 2*order)."""
    t = jnp.arange(n_steps)
    sins = jnp.stack([jnp.sin(2 * jnp.pi * i * t / period) for i in range(1, order + 1)], axis=1)
    coss = jnp.stack([jnp.cos(2 * jnp.pi * i * t / period) for i in range(1, order + 1)], axis=1)
    return jnp.concatenate([sins, coss], axis=1)


def fit_one_series(
    sales: np.ndarray,
    num_warmup: int = 100,
    num_samples: int = 100,
    num_chains: int = 4,
    seed: int = 0,
) -> dict:
    """Fit a local-level + weekly-seasonality state-space model to one series.

    Parameters
    ----------
    sales : np.ndarray
        1-D array of daily unit sales (n_steps,).
    num_warmup : int
        NUTS warmup iterations per chain.
    num_samples : int
        NUTS posterior samples per chain.
    num_chains : int
        Number of MCMC chains.
    seed : int
        PRNG seed for reproducibility.

    Returns
    -------
    dict
        Keys: ``posteriors`` (MCMC samples), ``response_norm`` (float),
        ``Z`` (design matrix), ``a0``, ``P0``.
    """
    # Clip zeros for log-transform, normalise
    sales_clipped = np.clip(sales, 1e-1, None).astype(np.float32)
    response_norm = float(sales_clipped.mean())
    y = jnp.array(np.log(sales_clipped / response_norm))

    n_steps = len(y)

    # Design matrix: intercept + weekly Fourier (order=3 → 6 cols) = 7 states
    weekly_seas = _make_fourier_series(n_steps, period=7.0, order=3)
    Z = jnp.concatenate([jnp.ones((n_steps, 1)), weekly_seas], axis=1)
    n_states = Z.shape[1]

    # Initial state
    a0 = jnp.zeros(n_states)
    P0 = jnp.ones(n_states)

    # sigma_q prior: [level, shared_seasonality]
    sigma_q_loc_prior = jnp.array([0.05, 0.01])
    sigma_q_scale_prior = jnp.array([0.05, 0.01])

    def _nuts_fn(a0, P0):
        sigma_h = numpyro.sample(
            "sigma_h",
            dist.TruncatedNormal(0.1, 1.0, high=1.0, low=1e-5),
        )
        sigma_q_raw = numpyro.sample(
            "sigma_q",
            dist.TruncatedNormal(
                sigma_q_loc_prior, sigma_q_scale_prior, high=0.1, low=1e-5,
            ),
        )
        # Expand: [level, seas1, seas2, …, seas6]
        n_seas = n_states - 1
        sigma_q = jnp.concatenate([sigma_q_raw[:1], jnp.repeat(sigma_q_raw[1:], n_seas)])

        lp, at, _, _, _, _ = kalman_filter_1d(
            a0=a0, P0=P0, sigma_h=sigma_h, sigma_q=sigma_q,
            y=y, Z=Z, logp=True,
        )
        numpyro.factor("lp", lp)
        numpyro.deterministic("at", at)
        numpyro.deterministic("mu", jnp.sum(Z * at, -1))

    rng_key = random.PRNGKey(seed)
    mcmc = MCMC(NUTS(_nuts_fn), num_warmup=num_warmup, num_samples=num_samples, num_chains=num_chains)
    mcmc.run(random.split(rng_key, 1)[0], a0, P0)

    return {
        "posteriors": mcmc.get_samples(),
        "response_norm": response_norm,
        "Z": Z,
        "a0": a0,
        "P0": P0,
    }


def predict_one_series(
    fit_result: dict,
    horizon: int = HORIZON,
) -> np.ndarray:
    """Generate point forecast (median) from a fitted single-series model.

    Parameters
    ----------
    fit_result : dict
        Output of ``fit_one_series()``.
    horizon : int
        Number of future steps to forecast.

    Returns
    -------
    np.ndarray
        Point forecasts of shape (horizon,).
    """
    posteriors = fit_result["posteriors"]
    response_norm = fit_result["response_norm"]

    # Last filtered state → naive constant-state forward projection
    # at: (n_samples, n_steps, n_states)
    at_samples = np.array(posteriors["at"])
    sigma_h_samples = np.array(posteriors["sigma_h"])

    n_samples = at_samples.shape[0]
    n_steps = at_samples.shape[1]

    # Build future Z (horizon steps continuing the Fourier clock)
    weekly_seas_future = np.array(
        _make_fourier_series(n_steps + horizon, period=7.0, order=3)
    )[n_steps:]
    Z_future = np.concatenate([np.ones((horizon, 1)), weekly_seas_future], axis=1)

    # Forward project: mu_t = Z_future @ a_last for each sample
    a_last = at_samples[:, -1, :]  # (n_samples, n_states)
    mu_future = a_last @ Z_future.T  # (n_samples, horizon)

    eps = np.random.default_rng(42).normal(0, sigma_h_samples[:, None], size=mu_future.shape)
    yhat_samples = np.exp(mu_future + eps) * response_norm

    # Median point forecast
    return np.median(yhat_samples, axis=0)

In [ ]:
# Pick the top-N series by total sales volume
N_DEMO = 1
total_sales = sales_matrix.sum(axis=1)
top_n_idx = np.argsort(total_sales)[::-1][:N_DEMO]

for rank, idx in enumerate(top_n_idx):
    print(f"  #{rank + 1}  {series_ids[idx]}  total={total_sales[idx]:,.0f}")

In [ ]:
# Fit and forecast each of the top-N series
demo_results = {}
for rank, idx in enumerate(top_n_idx):
    sid = series_ids[idx]
    print(f"\n[{rank + 1}/{N_DEMO}] Fitting {sid} …")
    fit_result = fit_one_series(sales_matrix[idx], num_warmup=100, num_samples=100, num_chains=4, seed=0)
    forecast = predict_one_series(fit_result, horizon=HORIZON)
    demo_results[sid] = {"idx": idx, "fit": fit_result, "forecast": forecast}

In [ ]:
# Plot last 90 days + 28-day forecast for each
tail = 90
fig, axes = plt.subplots(N_DEMO, 1, figsize=(12, 3 * N_DEMO), sharex=True, squeeze=False)
axes = axes[:, 0]
for ax, (sid, res) in zip(axes, demo_results.items()):
    actual = sales_matrix[res["idx"]]
    ax.plot(range(tail), actual[-tail:], label="Actual", alpha=0.7)
    ax.plot(range(tail, tail + HORIZON), res["forecast"], label="Forecast", linestyle="--", color="tomato")
    ax.axvline(tail, color="grey", linestyle=":", alpha=0.5)
    ax.set_title(sid, fontsize=9, loc="left")
    ax.set_ylabel("Units")
    ax.legend(fontsize=8)
axes[-1].set_xlabel("Day")
fig.suptitle(f"Top {N_DEMO} series by volume — SSP forecast", y=1.01)
plt.tight_layout()

## 4. Build Submission

The submission needs both **validation** (items ending `_validation`) and **evaluation** (`_evaluation`) rows — 60,980 rows total.

In [ ]:
def make_submission(
    predictions: pd.DataFrame,
    sample_submission: pd.DataFrame,
    output_path: Path | str = "submission.csv",
) -> pd.DataFrame:
    """Build a competition-ready submission CSV.

    Parameters
    ----------
    predictions : pd.DataFrame
        Forecast from ``predict()`` with ``id`` (validation ids) and F1–F28.
    sample_submission : pd.DataFrame
        Official sample_submission template to ensure correct row order.
    output_path : Path | str
        Where to write the CSV.

    Returns
    -------
    pd.DataFrame
        The final submission dataframe.
    """
    output_path = Path(output_path)
    f_cols = [f"F{i}" for i in range(1, HORIZON + 1)]

    # Validation rows: use predictions directly
    val_preds = predictions.copy()
    val_preds["id"] = val_preds["id"].str.replace("_validation", "_validation", regex=False)

    # Evaluation rows: duplicate predictions (same forecast for both horizons as placeholder)
    eval_preds = predictions.copy()
    eval_preds["id"] = eval_preds["id"].str.replace("_validation", "_evaluation", regex=False)

    full = pd.concat([val_preds, eval_preds], ignore_index=True)

    # Align to official submission order
    sub = sample_submission[["id"]].merge(full, on="id", how="left")

    assert sub.shape[0] == sample_submission.shape[0], (
        f"Row count mismatch: got {sub.shape[0]}, expected {sample_submission.shape[0]}"
    )
    assert not sub[f_cols].isna().any().any(), "Missing forecasts detected"

    sub.to_csv(output_path, index=False)
    print(f"Submission written to {output_path}  ({sub.shape[0]} rows)")
    return sub

In [ ]:
# Generate and save submission
sub = make_submission(predictions, submission, output_path="submission.csv")
sub.head()